# OceanScope AI — Full-Period Store Build (PRD_AI_ML §4.2, §11)

This notebook runs the **full-period streaming dataset build (1993–2021)** for OceanScope AI on Google Colab or Kaggle.

### Pipeline Design:
- **Unit of work:** 1 calendar year per step.
- **Stream $\to$ Reduce $\to$ Store:** Remote requests are fetched, reduced immediately to the 0.25° grid at the 15 confirmed GLORYS levels, and appended to the Zarr store.
- **No raw archive on disk:** Scratch disk stays $\le 20$ GB; Master store stays $\le 20$ GB (float16 copy $\le 12$ GB).
- **Resumable:** Skips finished years recorded in `built_years` attribute if interrupted.

## 1. Environment Setup & Dependencies

In [ ]:
# Install oceanscope_ml package and cloud streaming libraries
!pip install -q -e .
!pip install -q copernicusmarine cdsapi earthaccess

## 2. Load API Credentials from Cloud Secrets

Add your free provider credentials to **Google Colab Secrets** (🔑 icon on the left) or **Kaggle Secrets** (Add-ons $\to$ Secrets):
- `COPERNICUSMARINE_SERVICE_USERNAME` / `COPERNICUSMARINE_SERVICE_PASSWORD`
- `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD`
- `CDSAPI_URL` / `CDSAPI_KEY`
- `KAGGLE_USERNAME` / `KAGGLE_KEY` (for dataset upload)

In [ ]:
import os

# Try Google Colab userdata first, then Kaggle UserSecretsClient
try:
    from google.colab import userdata

    for key in [
        "COPERNICUSMARINE_SERVICE_USERNAME",
        "COPERNICUSMARINE_SERVICE_PASSWORD",
        "EARTHDATA_USERNAME",
        "EARTHDATA_PASSWORD",
        "CDSAPI_URL",
        "CDSAPI_KEY",
        "KAGGLE_USERNAME",
        "KAGGLE_KEY",
    ]:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        for key in [
            "COPERNICUSMARINE_SERVICE_USERNAME",
            "COPERNICUSMARINE_SERVICE_PASSWORD",
            "EARTHDATA_USERNAME",
            "EARTHDATA_PASSWORD",
            "CDSAPI_URL",
            "CDSAPI_KEY",
        ]:
            try:
                os.environ[key] = secrets.get_secret(key)
            except Exception:
                pass
    except ImportError:
        pass

print("✓ Credentials loaded into environment variables.")

## 3. Clone Repository & Setup Working Directory

In [ ]:
import sys
from pathlib import Path

# If running directly in cloned repo:
repo_root = Path.cwd()
ml_src = repo_root / "ml" / "src"
if ml_src.exists():
    sys.path.insert(0, str(ml_src))
    %cd ml
else:
    print("Clone the repo or upload the code to run in Colab/Kaggle:")
    # !git clone https://github.com/<user>/Ocean-Embed.git
    # %cd Ocean-Embed/ml
    # sys.path.insert(0, str(Path.cwd() / 'src'))

print("Python path updated.")

## 4. Run Dry-Run Inspection

Prints the planned remote transfers and storage growth table for 1993–2021 without downloading data.

In [ ]:
!python -m oceanscope_ml.data.build_store \
    --config configs/experiment.yaml \
    --years 1993-2021 \
    --dry-run

## 5. Execute Streaming Build (1993–2021)

Processes each year sequentially with scratch cleanup and auto-resumes if the session disconnects.

In [ ]:
!python -u -m oceanscope_ml.data.build_store \
    --config configs/experiment.yaml \
    --years 1993-2021 \
    --out-dir data/processed \
    --scratch-dir /tmp/oceanscope_scratch \
    --stream \
    --f16 \
    --finalize

## 6. Verify Store Schema & Integrity

In [ ]:
import json
from pathlib import Path

import xarray as xr

store_path = "data/processed/oceanscope_v1.zarr"
ds = xr.open_zarr(store_path)
print("=== Master Processed Zarr Store ===")
print(ds)
print("\nBuilt Years:", ds.attrs.get("built_years"))
stats_raw = ds.attrs.get("norm_stats", {})
stats_dict = json.loads(stats_raw) if isinstance(stats_raw, str) else stats_raw
print("Norm Stats Channels:", list(stats_dict.get("channels", {}).keys()))

fp16_path = "data/processed/oceanscope_v1_f16.zarr"
if Path(fp16_path).exists():
    ds_fp16 = xr.open_zarr(fp16_path)
    print("\n=== Float16 Training Store ===")
    print("thetao dtype:", ds_fp16["thetao"].dtype)
    sz_gb = sum(f.stat().st_size for f in Path(fp16_path).rglob("*") if f.is_file()) / (1024**3)
    print("Store footprint:", f"{sz_gb:.2f} GB")

## 7. Upload Float16 Store to Kaggle Dataset (for Phase 4 Training)

In [ ]:
import json
import os
from pathlib import Path

# Create dataset metadata for Kaggle API
dataset_dir = Path("data/processed/kaggle_upload")
dataset_dir.mkdir(parents=True, exist_ok=True)

meta = {
    "title": "OceanScope AI Processed Store v1",
    "id": f"{os.environ.get('KAGGLE_USERNAME', 'username')}/oceanscope-v1-fp16",
    "licenses": [{"name": "CC-BY-4.0"}],
}
with open(dataset_dir / "dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Upload folder ready. Run kaggle CLI to upload:")
!kaggle datasets create -p data/processed/kaggle_upload -r zip